In [7]:
# 导入必要的库
import pandas as pd
import numpy as np
import os
import random

# 设置随机种子以保证结果可复现
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Random seed set to {SEED}")

Random seed set to 42


In [8]:
# 读取数据
input_path = '../../DataFillter/sg/sg_filtered_cgm_data.csv'

if os.path.exists(input_path):
    print(f"Reading data from: {input_path}")
    df = pd.read_csv(input_path)
    df['time'] = pd.to_datetime(df['time'])
    print(f"Data loaded successfully. Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
else:
    print(f"Error: File not found at {input_path}")

Reading data from: ../../DataFillter/sg/sg_filtered_cgm_data.csv
Data loaded successfully. Shape: (101596, 5)
Columns: ['id', 'time', 'gl_sg', 'age', 'bmi']


In [9]:
# 确定要划分的受试者
unique_subjects = sorted(df['id'].unique())
total_subjects = len(unique_subjects)
print(f"Total unique subjects: {total_subjects}")

# 选取 10 个受试者作为 Served Set (用于迁移学习)
# 为了保证“均匀”且“每次结果一致”，我们在排序后的ID列表中进行等间隔采样
num_served = 10
if total_subjects <= num_served:
    print("Error: Not enough subjects to split.")
else:
    # 使用 np.linspace 生成等间隔的索引
    # 从 0 到 total_subjects-1，生成 num_served 个点，并转为整数索引
    indices = np.linspace(0, total_subjects - 1, num_served, dtype=int)
    
    # 根据索引获取 ID
    served_ids = [unique_subjects[i] for i in indices]
    served_ids = sorted(served_ids)
    
    # 剩余的作为 Train & Test Set
    train_test_ids = sorted(list(set(unique_subjects) - set(served_ids)))
    
    print(f"Selected {len(served_ids)} subjects for Served Set (Uniformly Spaced): {served_ids}")
    print(f"Remaining {len(train_test_ids)} subjects for Train & Test Set.")

Total unique subjects: 168
Selected 10 subjects for Served Set (Uniformly Spaced): [np.int64(2), np.int64(26), np.int64(48), np.int64(67), np.int64(95), np.int64(117), np.int64(142), np.int64(165), np.int64(185), np.int64(258)]
Remaining 158 subjects for Train & Test Set.


In [10]:
# 分割数据
df_served = df[df['id'].isin(served_ids)].copy()
df_train_test = df[df['id'].isin(train_test_ids)].copy()

print(f"Served Set shape: {df_served.shape}")
print(f"Train & Test Set shape: {df_train_test.shape}")

Served Set shape: (6967, 5)
Train & Test Set shape: (94629, 5)


In [11]:
# 重命名列名
# 将 'gl_sg' (或其他滤波列名) 重命名为 'gl'
target_col = 'gl'
filter_col = 'gl_sg' # 这里我们知道输入的是 S-G 滤波后的数据

if filter_col in df_served.columns:
    df_served.rename(columns={filter_col: target_col}, inplace=True)
    df_train_test.rename(columns={filter_col: target_col}, inplace=True)
    print(f"Renamed column '{filter_col}' to '{target_col}' in both datasets.")
else:
    print(f"Warning: Column '{filter_col}' not found. Available columns: {df_served.columns.tolist()}")

# 确保只保留需要的列 (可选，根据需求调整)
keep_cols = ['id', 'time', 'gl', 'age', 'bmi']
# 如果原始数据中有 residual 等列，可能不需要导出
df_served = df_served[keep_cols]
df_train_test = df_train_test[keep_cols]

print("Columns finalized.")
df_served.head()

Renamed column 'gl_sg' to 'gl' in both datasets.
Columns finalized.


,id,time,gl,age,bmi
0,2,2012-01-01 00:00:00,169.746732,42.0,30.0
1,2,2012-01-01 00:05:00,162.297852,42.0,30.0
2,2,2012-01-01 00:10:00,155.290688,42.0,30.0
3,2,2012-01-01 00:15:00,148.809452,42.0,30.0
4,2,2012-01-01 00:20:00,142.938357,42.0,30.0


In [12]:
# 导出数据

# 1. 导出 Served Set 到当前目录
output_served = 'served.csv'
df_served.to_csv(output_served, index=False)
print(f"Exported Served Set to: {os.path.abspath(output_served)}")

# 2. 导出 Train & Test Set 到 ../TrainTest/ 目录
output_train_test_dir = '../TrainTest'
if not os.path.exists(output_train_test_dir):
    os.makedirs(output_train_test_dir)
    print(f"Created directory: {output_train_test_dir}")

output_train_test = os.path.join(output_train_test_dir, 'TrainTest.csv')
df_train_test.to_csv(output_train_test, index=False)
print(f"Exported Train & Test Set to: {os.path.abspath(output_train_test)}")

Exported Served Set to: c:\Users\江一骏\Desktop\学习\HKU\dissertation\engineering\code\GlucosePrediction\src\DataSplit\Served\served.csv
Exported Train & Test Set to: c:\Users\江一骏\Desktop\学习\HKU\dissertation\engineering\code\GlucosePrediction\src\DataSplit\TrainTest\TrainTest.csv


## 总结

1.  **数据源**: 使用了 `sg_filtered_cgm_data.csv`。
2.  **划分**: 
    *   **Served Set**: 在排序后的受试者列表中**等间隔选取**了 10 个受试者。
    *   **Train & Test Set**: 包含剩余的所有受试者。
3.  **处理**: 将血糖列统一重命名为 `gl`。
4.  **输出**:
    *   `src/DataSplit/Served/served.csv`
    *   `src/DataSplit/TrainTest/TrainTest.csv`